In [1]:
# 6-25-2026

In [2]:
import xarray as xr
import numpy as np
import pandas as pd

In [3]:
ds_path = "../data/seasfire_pyromes_ecoregions.zarr"
domains_path = "../data/ecoregion_domains.zarr"

In [4]:
ds = xr.open_zarr(ds_path, consolidated=True)
domains = xr.open_zarr(domains_path, consolidated=True)

In [5]:
ds = ds.drop_vars(["biomes", "ecoregion"])

In [6]:
ds["domain_id"] = domains["domain_id"] # add in domain id to ds, same spatial res

In [7]:
print(list(ds.data_vars))

['area', 'cams_co2fire', 'cams_frpfire', 'drought_code_max', 'drought_code_mean', 'fcci_ba', 'fcci_ba_valid_mask', 'fcci_fraction_of_burnable_area', 'fcci_fraction_of_observed_area', 'fcci_number_of_patches', 'fwi_max', 'fwi_mean', 'gwis_ba', 'gwis_ba_valid_mask', 'lai', 'lccs_class_1', 'lccs_class_2', 'lccs_class_3', 'lccs_class_4', 'lccs_class_6', 'lccs_class_7', 'lsm', 'lst_day', 'ndvi', 'pop_dens', 'pyrome', 'rel_hum', 'skt', 'ssr', 'ssrd', 'sst', 'swvl1', 'swvl2', 'swvl3', 'swvl4', 't2m_max', 't2m_mean', 't2m_min', 'tp', 'vpd', 'ws10', 'domain_id']


In [ ]:
"""
will use these features mean/std/90th:

vpd
windspeed
t2m_max
ndvi
tp
fwi_mean
swvl 1 and 4 only
burned area (gwis)
frp


last two features (ba, frp) stats will only be collected for fire cells (ba>0)
all features will skip over missing values
"""

'\nwill use these features mean/std/90th:\n\nvpd\nwindspeed\nt2m_max\nndvi\ntp\nfwi_mean\nswvl 1 and 4 only\nburned area (gwis)\nfrp\n\n\nlast two features (ba, frp) stats will only be collected for fire cells (ba>0)\n'

In [9]:
features = ["vpd", "ws10", "t2m_max", "ndvi", "tp", "fwi_mean", "swvl1", "swvl4"]

In [10]:
# load domain array and get valid domain ids (exclude -1)
domain_arr = ds["domain_id"].values
unique_domains = np.unique(domain_arr)
unique_domains = unique_domains[unique_domains != -1]

In [11]:
# init results dict keyed by domain id
results = {int(d): {} for d in unique_domains}

In [12]:
for var in features:
    print(f"processing {var}")
    data = ds[var].values  # load once per variable, then reuse across all domains

    for domain in unique_domains:
        spatial_mask = (domain_arr == domain)  # bool mask over lat, lon

        if data.ndim == 3:
            # time, lat, lon: select all timesteps for cells in this domain
            vals = data[:, spatial_mask].ravel()
        elif data.ndim == 2:
            vals = data[spatial_mask].ravel()

        vals = vals[~np.isnan(vals)]

        results[int(domain)][f"{var}_mean"] = np.mean(vals)
        results[int(domain)][f"{var}_std"] = np.std(vals)
        results[int(domain)][f"{var}_p90"] = np.percentile(vals, 90)

    del data  # free memory before loading next variable
    print(f"done getting satts for {var}")

# takes ~2 min

processing vpd...
done getting satts for vpd
processing ws10...
done getting satts for ws10
processing t2m_max...
done getting satts for t2m_max
processing ndvi...
done getting satts for ndvi
processing tp...
done getting satts for tp
processing fwi_mean...
done getting satts for fwi_mean
processing swvl1...
done getting satts for swvl1
processing swvl4...
done getting satts for swvl4


In [13]:
domain_stats = pd.DataFrame.from_dict(results, orient="index")
domain_stats.index.name = "domain_id"

In [15]:
domain_stats.head(10)

,vpd_mean,vpd_std,vpd_p90,ws10_mean,ws10_std,ws10_p90,t2m_max_mean,t2m_max_std,t2m_max_p90,ndvi_mean,...,tp_p90,fwi_mean_mean,fwi_mean_std,fwi_mean_p90,swvl1_mean,swvl1_std,swvl1_p90,swvl4_mean,swvl4_std,swvl4_p90
domain_id,,,,,,,,,,,,,,,,,,,,,
0,11.229313,5.895861,19.746269,2.516127,1.052111,3.968697,302.433167,4.790887,307.460022,0.641714,...,80.862717,13.352995,15.344999,38.503906,0.324444,0.118173,0.477633,0.349587,0.107862,0.479085
1,6.742952,3.144529,10.522034,1.406203,0.511182,2.132592,301.806488,4.735351,305.513702,0.796118,...,106.794235,2.711859,6.101745,8.196289,0.412610,0.070826,0.490699,0.428787,0.055336,0.499585
2,21.599873,10.565010,36.654785,4.066474,0.951055,5.291819,303.896240,6.380805,311.617920,0.242577,...,20.986458,50.144970,22.770884,78.126465,0.087747,0.078288,0.201105,0.156024,0.070416,0.257699
3,4.211150,2.716757,7.958644,3.166607,1.505368,5.099028,286.087921,10.954839,299.418915,0.550528,...,73.198547,1.899023,2.517217,5.039062,0.305262,0.101840,0.399886,0.325720,0.103028,0.411259
4,2.564834,3.033652,6.938553,3.300411,1.380530,5.114551,273.420776,16.251247,293.801086,0.315671,...,29.893572,2.741655,5.352413,8.728906,0.335655,0.096505,0.426583,0.365052,0.094277,0.447848
5,10.265495,5.699432,17.502420,2.046542,0.978449,3.276847,301.659851,5.860431,307.498688,0.642368,...,102.891647,10.326067,13.448431,31.254681,0.353536,0.110351,0.485129,0.400621,0.084440,0.491259
6,3.370416,3.105505,7.804835,3.172674,1.288420,4.654522,280.814453,12.255744,296.831268,0.373939,...,48.522751,3.143916,5.783900,9.461330,0.296224,0.087604,0.392211,0.309804,0.079603,0.394887
7,11.621094,8.784235,24.302675,3.065614,1.245384,4.709380,295.900330,8.041678,307.009216,0.308755,...,26.284737,28.404888,22.160028,58.745880,0.159476,0.115823,0.331174,0.194564,0.072165,0.286967
8,7.512378,2.146205,10.056996,1.712309,1.128670,3.099993,302.636597,2.228121,305.122589,0.792586,...,126.542206,2.085299,5.289259,5.523438,0.354161,0.122412,0.479011,0.371558,0.116982,0.488031


In [16]:
fire_features = ["gwis_ba", "cams_frpfire"]
fire_results = {int(d): {} for d in unique_domains}
# now for ba and frp, these are only fire cells

In [17]:
for var in fire_features:
    print(f"processing {var}")
    data = ds[var].values
    ba = ds["gwis_ba"].values  # fire mask reference

    for domain in unique_domains:
        spatial_mask = (domain_arr == domain)

        if data.ndim == 3:
            domain_data = data[:, spatial_mask]
            domain_ba = ba[:, spatial_mask]
        elif data.ndim == 2:
            domain_data = data[spatial_mask]
            domain_ba = ba[spatial_mask]

        fire_mask = domain_ba > 0
        vals = domain_data[fire_mask].ravel()
        vals = vals[~np.isnan(vals)]

        fire_results[int(domain)][f"{var}_mean"] = np.mean(vals) if len(vals) > 0 else np.nan
        fire_results[int(domain)][f"{var}_std"] = np.std(vals) if len(vals) > 0 else np.nan
        fire_results[int(domain)][f"{var}_p90"] = np.percentile(vals, 90) if len(vals) > 0 else np.nan

    del data

processing gwis_ba
processing cams_frpfire


In [18]:
fire_stats = pd.DataFrame.from_dict(fire_results, orient="index")
fire_stats.index.name = "domain_id"

domain_stats = domain_stats.join(fire_stats)

In [19]:
domain_stats.head()

,vpd_mean,vpd_std,vpd_p90,ws10_mean,ws10_std,ws10_p90,t2m_max_mean,t2m_max_std,t2m_max_p90,ndvi_mean,...,swvl1_p90,swvl4_mean,swvl4_std,swvl4_p90,gwis_ba_mean,gwis_ba_std,gwis_ba_p90,cams_frpfire_mean,cams_frpfire_std,cams_frpfire_p90
domain_id,,,,,,,,,,,,,,,,,,,,,
0,11.229313,5.895861,19.746269,2.516127,1.052111,3.968697,302.433167,4.790887,307.460022,0.641714,...,0.477633,0.349587,0.107862,0.479085,547.384094,1932.254639,1112.033691,0.067789,0.469590,0.097816
1,6.742952,3.144529,10.522034,1.406203,0.511182,2.132592,301.806488,4.735351,305.513702,0.796118,...,0.490699,0.428787,0.055336,0.499585,338.661652,1157.090332,705.891968,0.063071,0.392797,0.108795
2,21.599873,10.565010,36.654785,4.066474,0.951055,5.291819,303.896240,6.380805,311.617920,0.242577,...,0.201105,0.156024,0.070416,0.257699,3360.493408,8181.072266,8625.163086,0.162555,0.925012,0.219152
3,4.211150,2.716757,7.958644,3.166607,1.505368,5.099028,286.087921,10.954839,299.418915,0.550528,...,0.399886,0.325720,0.103028,0.411259,328.766846,739.654419,901.446350,0.041860,0.274789,0.000000
4,2.564834,3.033652,6.938553,3.300411,1.380530,5.114551,273.420776,16.251247,293.801086,0.315671,...,0.426583,0.365052,0.094277,0.447848,669.052063,2307.770020,1315.654907,0.185203,1.571907,0.090595


In [20]:
domain_stats.to_csv("simple_domain_stats.csv")